# 04 - Uncapped Category Counts

**Project:** Research Paper Discovery and Trend Analysis (Phase 1)  
**Owner:** Person A (data pipeline)

`papers_raw.parquet` is already capped at 40,000 papers per category, so it
cannot show the original arXiv class imbalance. This notebook streams the
original **Cornell University arXiv snapshot** (the input used by step A1) and
counts papers per primary category and year **before the cap**.

**Input requirement:** attach the Kaggle dataset
[`Cornell-University/arxiv`](https://www.kaggle.com/datasets/Cornell-University/arxiv)
before running. Do not use `papers_raw.parquet` as the input here.

The reusable logic lives in `src/uncapped_counts.py`; this notebook imports and
calls it.


## Setup

This works on Kaggle and locally. On Kaggle the notebook clones the project
repository if it is not already available, then writes its outputs to
`/kaggle/working`. Locally it expects to run from the repository and writes to
`results/`.


In [ ]:
import os, sys, subprocess
from pathlib import Path

import pandas as pd
from IPython.display import display

ON_KAGGLE = Path("/kaggle/input").exists()
REPO_URL = "https://github.com/MohammadIbraheem71/research_paper_discovery.git"

if ON_KAGGLE:
    REPO_DIR = Path("/kaggle/working/research_paper_discovery")
else:
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# On Kaggle the notebook may be uploaded on its own, so clone the repo if needed.
if ON_KAGGLE and not (REPO_DIR / "src" / "uncapped_counts.py").exists():
    if (REPO_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

sys.path.insert(0, str(REPO_DIR))
OUT_DIR = Path("/kaggle/working") if ON_KAGGLE else REPO_DIR / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("repo       :", REPO_DIR)
print("output dir :", OUT_DIR)


## Locate the raw snapshot

`find_snapshot()` searches `/kaggle/input` on Kaggle and `data/` locally for
`arxiv-metadata-oai-snapshot.json`.


In [ ]:
from src.uncapped_counts import find_snapshot, count_uncapped, build_tables

snapshot = find_snapshot()
print("raw snapshot:", snapshot)


## Count papers without the cap

The filter is the same as step A1: primary category in `cs.LG`, `cs.CL`,
`cs.CV`, and first-version year between 2018 and 2024. The 40,000-per-category
cap is **not** applied.


In [ ]:
by_category, by_category_year, by_year, scanned, skipped = count_uncapped(snapshot)
category_df, year_df, pivot = build_tables(by_category, by_category_year)

total_uncapped = int(category_df['papers'].sum())
assert total_uncapped > 0, (
    'Uncapped counts are zero. Check the snapshot and date parsing before saving these CSVs.'
)

print(f"scanned lines                  : {scanned:,}")
print(f"kept after category/year filter: {total_uncapped:,}")
print(f"malformed / skipped lines      : {skipped:,}")


## Papers per category before the cap

`papers` is the real number of papers in the category for 2018–2024.
`after_cap` and `removed_by_cap` show exactly what the balancing step did.


In [ ]:
display(category_df)


## Papers per category and year

Rows are years, columns are categories. The `total` column is the number of
kept papers in that year across the three categories.


In [ ]:
display(pivot)


## Save the tables

On Kaggle the files are written to `/kaggle/working`; locally they are written
to `results/`. These are the files to share with Person B for the EDA
class-balance section.


In [ ]:
category_df.to_csv(OUT_DIR / "uncapped_counts.csv", index=False)
year_df.to_csv(OUT_DIR / "uncapped_counts_by_year.csv", index=False)
pivot.to_csv(OUT_DIR / "uncapped_counts_pivot.csv")
pd.Series(by_year, name="papers").sort_index().to_csv(
    OUT_DIR / "uncapped_totals_by_year.csv"
)

print("saved:")
for name in [
    "uncapped_counts.csv",
    "uncapped_counts_by_year.csv",
    "uncapped_counts_pivot.csv",
    "uncapped_totals_by_year.csv",
]:
    print(" -", OUT_DIR / name)


## Summary for Person B

The three target categories are **not** equally large on arXiv. The project
sample is balanced only because of the 40,000-per-category cap. Use these
counts to explain the original imbalance in the EDA class-balance section,
together with the uncapped-over-time table.


In [ ]:
total_uncapped = int(category_df["papers"].sum())
total_capped = int(category_df["after_cap"].sum())

print(f"Uncapped papers in scope          : {total_uncapped:,}")
print(f"Papers kept after the 40k cap     : {total_capped:,}")
print(f"Papers removed by the cap         : {total_uncapped - total_capped:,}")
print("\nShare of the uncapped scope:")
print((category_df["papers"] / total_uncapped * 100).round(2).to_string())
